# Experiment 03: RAG Orchestration with Chroma & LCEL
- **Goal:** Implement an end-to-end Retrieval-Augmented Generation (RAG) pipeline and validate orchestration using the new LangChain Expression Language (LCEL).
- **Components:** `Docx2txtLoader`, `RecursiveCharacterTextSplitter`, `Chroma`, `OpenAIEmbeddings`, `LCEL`.
- **Method:**
    1. Load external documents (`.docx`) and split them into chunks (Size: 1500, Overlap: 200).
    2. Vectorize text using OpenAI Embeddings and persist in a local **Chroma** database.
    3. Construct a declarative chain: `Retriever | Prompt | LLM | OutputParser`.
- **Key Observation:** Transitioned from legacy `RetrievalQA` to modern LCEL for better modularity and transparency in data flow.

1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰 수 초과로 답변을 생성하지 못할 수 있고
    - 문서가(input 이) 길면 답변 생성이 오래 걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
# pip install docx2txt==0.9 langchain-community==0.4.1 langchain-text-splitters==1.1.2 langchain-chroma==1.1.0

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

database = Chroma.from_documents(
    documents=document_list,
    embedding=embedding,
    collection_name='chroma-tax',
    persist_directory='./chroma_db'
)

In [3]:
query = '소득세법에서 말하는 근로소득의 정의가 무엇인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [4]:
retrieved_docs

[Document(id='f3869c36-ddfb-42ca-b1c1-ef246d03a834', metadata={'source': './tax.docx'}, page_content='21. 제1호부터 제20호까지의 규정에 따른 소득과 유사한 소득으로서 영리를 목적으로 자기의 계산과 책임 하에 계속적ㆍ반복적으로 행하는 활동을 통하여 얻는 소득\n\n② 사업소득금액은 해당 과세기간의 총수입금액에서 이에 사용된 필요경비를 공제한 금액으로 하며, 필요경비가 총수입금액을 초과하는 경우 그 초과하는 금액을 “결손금”이라 한다.\n\n③ 제1항 각 호에 따른 사업의 범위에 관하여는 이 법에 특별한 규정이 있는 경우 외에는 「통계법」 제22조에 따라 국가데이터처장이 고시하는 한국표준산업분류에 따르고, 그 밖의 사업소득의 범위에 관하여 필요한 사항은 대통령령으로 정한다.<개정 2025. 10. 1.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제20조(근로소득) ① 근로소득은 해당 과세기간에 발생한 다음 각 호의 소득으로 한다. <개정 2016. 12. 20., 2024. 12. 31.>\n\n1. 근로를 제공함으로써 받는 봉급ㆍ급료ㆍ보수ㆍ세비ㆍ임금ㆍ상여ㆍ수당과 이와 유사한 성질의 급여\n\n2. 법인의 주주총회ㆍ사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득\n\n3. 「법인세법」에 따라 상여로 처분된 금액\n\n4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득\n\n5. 종업원등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외한다)\n\n6. 사업자나 법인이 생산ㆍ공급하는 재화 또는 용역을 그 사업자나 법인(「독점규제 및 공정거래에 관한 법률」에 따른 계열회사를 포함한다)의 사업장에 종사하는 임원등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원등이 얻는 이익\n\n② 근로소득금액은 제1항 각 호의 소득의 금액의 합

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

prompt = f"""[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다
- [Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrieved_docs}

Question: {query}
"""

ai_message = llm.invoke(prompt)

In [6]:
ai_message.content

'소득세법에서 근로소득의 정의는 근로를 제공함으로써 받는 소득을 의미합니다. 구체적으로는 봉급, 급료, 보수, 세비, 임금, 상여, 수당과 이와 유사한 성질의 급여를 포함합니다. 또한, 법인의 주주총회, 사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득, 법인세법에 따라 상여로 처분된 금액, 그리고 퇴직함으로써 받는 소득 중 퇴직소득에 속하지 않는 소득 등이 포함됩니다. 또한, 직무발명보상금 및 특정 조건 하에서 시가보다 낮은 가격으로 공급받아 얻는 이익도 포함됩니다. 근로소득금액은 이러한 소득의 합계에서 근로소득공제를 적용한 금액으로 결정됩니다.'

In [ ]:
#pip install langchain==1.2.15

In [7]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

In [8]:
prompt.messages

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [9]:
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o')
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

database = Chroma(
    collection_name='chroma-tax',
    persist_directory='./chroma_db',
    embedding_function=embedding
)   

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = database.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
query = '소득세법에서 말하는 근로소득의 정의가 무엇인가요?'
response = rag_chain.invoke(query)

In [11]:
response

'소득세법에서의 근로소득은 근로를 제공함으로써 받는 봉급, 급료, 임금, 상여, 수당 등과 이와 유사한 성질의 급여를 의미합니다. 또한, 법인의 주주총회 등에서 결의에 따라 상여로 받는 소득이나 퇴직 시 받는 소득 중 퇴직소득에 속하지 않는 소득도 포함됩니다. 추가로, 직무발명보상금이나 임원 등이 시가보다 낮은 가격으로 제공받는 이익 등도 근로소득에 해당합니다.'